In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
import numpy as np
import torch
from dataclasses import dataclass
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [ ]:
@dataclass
class Config:
    model_name: str = "distilbert-base-uncased"
    dataset_name: str = "imdb"
    train_samples: int = 4000
    test_samples: int = 1000
    max_length: int = 256
    batch_size: int = 16
    learning_rate: float = 2e-5
    epochs: int = 2
    weight_decay: float = 0.01
    seed: int = 42
    output_dir: str = "distilbert-imdb"

cfg = Config()
set_seed(cfg.seed)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
cfg.dataset_name = "stanfordnlp/imdb"
print("Dataset name is now:", cfg.dataset_name)

Dataset name is now: stanfordnlp/imdb


In [ ]:
raw = load_dataset(cfg.dataset_name)
print(raw)

sample = raw["train"][0]
print("\nLabel:", sample["label"], "(0 = negative, 1 = positive)")
print("Text:", sample["text"][:300], "...")

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

Label: 0 (0 = negative, 1 = positive)
Text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h ...


In [ ]:
train_ds = raw["train"].shuffle(seed=cfg.seed).select(range(cfg.train_samples))
test_ds  = raw["test"].shuffle(seed=cfg.seed).select(range(cfg.test_samples))

print("Train size:", len(train_ds))
print("Test size :", len(test_ds))
print("Train label counts:", np.bincount(train_ds["label"]))

Train size: 4000
Test size : 1000
Train label counts: [2015 1985]


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

text = "This movie was absolutely fantastic!"
encoded = tokenizer(text)

print("Input IDs     :", encoded["input_ids"])
print("Attention mask:", encoded["attention_mask"])
print("Tokens        :", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Input IDs     : [101, 2023, 3185, 2001, 7078, 10392, 999, 102]
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1]
Tokens        : ['[CLS]', 'this', 'movie', 'was', 'absolutely', 'fantastic', '!', '[SEP]']


In [ ]:
def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=cfg.max_length)

train_tok = train_ds.map(tokenize_fn, batched=True)
test_tok  = test_ds.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_tok)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4000
})


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    cfg.model_name,
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
)
model.to(device)
print("Parameters:", round(sum(p.numel() for p in model.parameters()) / 1e6, 1), "million")

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 67.0 million


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    learning_rate=cfg.learning_rate,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size * 2,
    num_train_epochs=cfg.epochs,
    weight_decay=cfg.weight_decay,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=cfg.seed,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.337593,0.285388,0.876000,0.872690
2,0.166274,0.298068,0.881000,0.879433


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.312517749786377, metrics={'train_runtime': 76.2819, 'train_samples_per_second': 104.874, 'train_steps_per_second': 6.555, 'total_flos': 529869594624000.0, 'train_loss': 0.312517749786377, 'epoch': 2.0})

In [ ]:
results = trainer.evaluate()
print(results)

preds_output = trainer.predict(test_tok)
preds = np.argmax(preds_output.predictions, axis=-1)
labels = preds_output.label_ids

print("\nConfusion matrix (rows = true, cols = predicted):")
print(confusion_matrix(labels, preds))

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.166274,0.298068,2,0.881000,0.879433


{'eval_loss': 0.2980678081512451, 'eval_accuracy': 0.881, 'eval_f1': 0.8794326241134752}



Confusion matrix (rows = true, cols = predicted):
[[447  65]
 [ 54 434]]


In [ ]:
from transformers import pipeline

clf = pipeline(
    "sentiment-analysis",
    model=trainer.model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

tests = [
    "I loved every minute of this film, the acting was brilliant!,",
    "What a waste of time. The plot made no sense and I almost fell asleep.",
    "It was okay, nothing special but not terrible either.",
    "Not bad at all, I was pleasantly surprised.",
    "The movie was not good.",
]

for t in tests:
    out = clf(t)[0]
    print(f"{out['label']:9} ({out['score']:.2f})  ->  {t}")

NEGATIVE  (0.77)  ->  I loved every minute of this film, the acting was brilliant!,The movie was not good overall
NEGATIVE  (0.97)  ->  What a waste of time. The plot made no sense and I almost fell asleep.
NEGATIVE  (0.53)  ->  It was okay, nothing special but not terrible either.
NEGATIVE  (0.51)  ->  Not bad at all, I was pleasantly surprised.
NEGATIVE  (0.96)  ->  The movie was not good.


In [ ]:
trainer.save_model("my-distilbert-sentiment")
tokenizer.save_pretrained("my-distilbert-sentiment")

# Optional: reload later like this
# clf = pipeline("sentiment-analysis", model="my-distilbert-sentiment")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('my-distilbert-sentiment/tokenizer_config.json',
 'my-distilbert-sentiment/tokenizer.json')

In [ ]:
!zip -r my-distilbert-sentiment.zip my-distilbert-sentiment
from google.colab import files
files.download("my-distilbert-sentiment.zip")

  adding: my-distilbert-sentiment/ (stored 0%)
  adding: my-distilbert-sentiment/tokenizer_config.json (deflated 43%)
  adding: my-distilbert-sentiment/tokenizer.json (deflated 71%)
  adding: my-distilbert-sentiment/config.json (deflated 50%)
  adding: my-distilbert-sentiment/training_args.bin (deflated 53%)
  adding: my-distilbert-sentiment/model.safetensors (deflated 8%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>